# GNU Radio IIR low-pass filter for synthetic IQ data

This notebook builds a type-correct GNU Radio flowgraph. The complex IQ stream is split into float I and Q streams, filtered independently with `single_pole_iir_filter_ff`, and recombined before it reaches the complex sink.

In [ ]:
import numpy as np
from gnuradio import blocks, gr

In [ ]:
N, L = 5, 1000
SEED = 42
ALPHA = 0.1
rng = np.random.default_rng(SEED)
i_samples = rng.standard_normal((N, L)).astype(np.float32)
q_samples = rng.standard_normal((N, L)).astype(np.float32)
input_iq = i_samples + 1j * q_samples
input_flat = input_iq.reshape(-1).astype(np.complex64)
print(f'Input shape: {input_iq.shape}')
print(f'Input samples: {input_flat.size}')
print(f'Input mean power: {np.mean(np.abs(input_flat) ** 2):.6f}')

## Flowgraph topology

`vector_source_c -> complex_to_float -> two single_pole_iir_filter_ff blocks -> float_to_complex -> vector_sink_c`

The two scalar IIR filters are necessary because `single_pole_iir_filter_ff` accepts float samples, not complex samples.

In [ ]:
class IIRFilterFlowgraph(gr.top_block):
    def __init__(self, samples, alpha):
        super().__init__()
        self.source = blocks.vector_source_c(samples.tolist(), repeat=False)
        self.complex_to_float = blocks.complex_to_float(vlen=1)
        self.iir_i = blocks.single_pole_iir_filter_ff(alpha, 1)
        self.iir_q = blocks.single_pole_iir_filter_ff(alpha, 1)
        self.float_to_complex = blocks.float_to_complex(vlen=1)
        self.sink = blocks.vector_sink_c()

        self.connect(self.source, self.complex_to_float)
        self.connect((self.complex_to_float, 0), self.iir_i)
        self.connect((self.complex_to_float, 1), self.iir_q)
        self.connect(self.iir_i, (self.float_to_complex, 0))
        self.connect(self.iir_q, (self.float_to_complex, 1))
        self.connect(self.float_to_complex, self.sink)

In [ ]:
flowgraph = IIRFilterFlowgraph(input_flat, ALPHA)
flowgraph.run()
output_flat = np.asarray(flowgraph.sink.data(), dtype=np.complex64)
output_iq = output_flat.reshape(N, L)
print(f'Output shape: {output_iq.shape}')
print(f'Output samples: {output_flat.size}')
print(f'Output mean power: {np.mean(np.abs(output_flat) ** 2):.6f}')
print(f'Input standard deviation: {np.std(input_flat):.6f}')
print(f'Output standard deviation: {np.std(output_flat):.6f}')

In [ ]:
input_power = np.mean(np.abs(input_flat) ** 2)
output_power = np.mean(np.abs(output_flat) ** 2)
assert output_flat.size == input_flat.size
assert output_iq.shape == input_iq.shape
assert not np.allclose(output_flat, input_flat)
assert output_power < input_power
print('PASS: output length matches input length')
print('PASS: output shape matches input shape')
print('PASS: IIR filtering changes the signal and reduces mean power')